In [14]:
import pandas as pd
import numpy as np
!pip install pm4py
import pm4py

CSV_PATH = "P2P.csv"


#1-GET AN EVENT LOG

#An event log needs 3 things at minimum:
#Case ID  -> which process instance an event belongs to (Order_ID)
#Activity -> what happened (Activity)
#Timestamp-> when it happened (Timestamp)

df = pd.read_csv(CSV_PATH)
print("Raw columns:", df.columns.tolist())


Raw columns: ['Unnamed: 0', 'Order_ID', 'Activity', 'Timestamp']


In [15]:
#2-IMPORT & CLEAN
#Drop junk columns
#Convert timestamp column to real datetime objects
#Sort events within each case chronologically (critical! logs are often
#exported in a random or partially-sorted order)
#Rename columns to pm4py's expected standard names

df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values(["Order_ID", "Timestamp"]).reset_index(drop=True)

#Check duplicate events (same case, activity, timestamp)
dupes = df.duplicated(subset=["Order_ID", "Activity", "Timestamp"]).sum()
print(f"Duplicate events found: {dupes}")

#is Case ID actually unique?
df = df.rename(columns={
    "Order_ID": "case:concept:name",
    "Activity": "concept:name",
    "Timestamp": "time:timestamp",
})

event_log = pm4py.format_dataframe(
    df,
    case_id="case:concept:name",
    activity_key="concept:name",
    timestamp_key="time:timestamp",
)

#How many unique case have?
unique_cases = df['case:concept:name'].nunique()
print(f"Total unique cases: {unique_cases}")

#How many activities each case have
case_counts = df['case:concept:name'].value_counts()
print("Top 5 case that have most activities")
print(case_counts.head())
print("Cases have only 1 activity", (case_counts == 1).sum())

# Is there any null in Case ID column
missing_cases = df['case:concept:name'].isna().sum()
print(f"NaN Case ID: {missing_cases}")


Duplicate events found: 0
Total unique cases: 934
Top 5 case that have most activities
case:concept:name
ORD0932    9
ORD0001    9
ORD0914    9
ORD0913    9
ORD0911    9
Name: count, dtype: int64
Cases have only 1 activity 0
NaN Case ID: 0


In [16]:
#3-DESCRIPTIVE STATS

n_cases = df["case:concept:name"].nunique()
n_events = len(df)
n_activities = df["concept:name"].nunique()
events_per_case = df.groupby("case:concept:name").size()

print("\n=== STEP 3: Descriptive stats ===")
print(f"Cases:      {n_cases}")
print(f"Events:     {n_events}")
print(f"Activities: {n_activities}")
print(f"Events per case -> min {events_per_case.min()}, "
      f"max {events_per_case.max()}, mean {events_per_case.mean():.2f}")

case_durations = pm4py.get_all_case_durations(event_log)  # in seconds
print(f"\nCase duration (days) -> "
      f"mean {np.mean(case_durations)/86400:.2f}, "
      f"median {np.median(case_durations)/86400:.2f}, "
      f"max {np.max(case_durations)/86400:.2f}")


=== STEP 3: Descriptive stats ===
Cases:      934
Events:     7782
Activities: 9
Events per case -> min 6, max 9, mean 8.33

Case duration (days) -> mean 85.77, median 40.07, max 363.38


In [20]:

#4-DISCOVER THE PROCESS MODEL
# (a) Directly-Follows Graph (DFG) — shows raw A->B frequencies
# (b) Inductive Miner — formal, sound process model
print("\n=== STEP 4: Process discovery ===")

# --- (a) DFG ---
dfg, start_activities, end_activities = pm4py.discover_dfg(event_log)
print("\nTop 10 directly-follows relations:")
for edge, freq in sorted(dfg.items(), key=lambda x: -x[1])[:10]:
    print(f"  {edge[0]} -> {edge[1]} : {freq}")

print("\nStart activities:", start_activities)
print("End activities:", end_activities)

pm4py.save_vis_dfg(dfg, start_activities, end_activities, "p2p_dfg.png")

#(b) Inductive Miner -> Petri net
net, initial_marking, final_marking = pm4py.discover_petri_net_inductive(event_log)
pm4py.save_vis_petri_net(net, initial_marking, final_marking, "p2p_petri_net.png")
print("\nPetri net discovered and saved to p2p_petri_net.png")


=== STEP 4: Process discovery ===

Top 10 directly-follows relations:
  Approve Purchase Requisition -> Create Purchase Order : 841
  Create Purchase Order -> Send Purchase Order to Supplier : 819
  Create Purchase Requisition -> Approve Purchase Requisition : 785
  Verify Goods Receipt -> Create Invoice : 780
  Receive Goods -> Verify Goods Receipt : 779
  Send Purchase Order to Supplier -> Receive Goods : 778
  Approve Invoice -> Make Payment : 769
  Create Invoice -> Approve Invoice : 723
  Make Payment -> Create Purchase Requisition : 139
  Create Invoice -> Make Payment : 94

Start activities: {'Create Purchase Requisition': 708, 'Approve Purchase Requisition': 95, 'Send Purchase Order to Supplier': 31, 'Create Purchase Order': 23, 'Approve Invoice': 30, 'Verify Goods Receipt': 16, 'Create Invoice': 15, 'Make Payment': 8, 'Receive Goods': 8}
End activities: {'Make Payment': 740, 'Create Purchase Requisition': 23, 'Create Purchase Order': 31, 'Approve Purchase Requisition': 8, 'Cr

In [19]:
#5-VARIANT ANALYSIS
# A "variant" is a unique sequence of activities. Group cases by variant to see
# the dominant "happy path" vs. rarer exception paths.

print("\n=== STEP 5: Variant analysis ===")

variants = pm4py.get_variants(event_log)                    # {variant_tuple: count}
sorted_variants = sorted(variants.items(), key=lambda x: x[1], reverse=True)

print(f"Number of unique variants: {len(sorted_variants)}")
for i, (variant, count) in enumerate(sorted_variants[:5]):
    pct = count / n_cases * 100
    print(f"\nVariant {i+1} ({count} cases, {pct:.1f}%):")
    print("  " + " -> ".join(variant))



=== STEP 5: Variant analysis ===
Number of unique variants: 34

Variant 1 (373 cases, 39.9%):
  Create Purchase Requisition -> Approve Purchase Requisition -> Create Purchase Order -> Send Purchase Order to Supplier -> Receive Goods -> Verify Goods Receipt -> Create Invoice -> Approve Invoice -> Make Payment

Variant 2 (63 cases, 6.7%):
  Create Purchase Requisition -> Approve Purchase Requisition -> Create Purchase Order -> Send Purchase Order to Supplier -> Receive Goods -> Verify Goods Receipt -> Create Invoice -> Make Payment

Variant 3 (63 cases, 6.7%):
  Approve Purchase Requisition -> Create Purchase Order -> Send Purchase Order to Supplier -> Receive Goods -> Verify Goods Receipt -> Create Invoice -> Approve Invoice -> Make Payment

Variant 4 (56 cases, 6.0%):
  Create Purchase Requisition -> Approve Purchase Requisition -> Create Purchase Order -> Send Purchase Order to Supplier -> Verify Goods Receipt -> Create Invoice -> Approve Invoice -> Make Payment

Variant 5 (40 cases,

In [21]:
#6-PERFORMANCE ANALYSIS (BOTTLENECKS)
#Same idea as the DFG, but instead of counting frequency, measure average
#(or median,max) time between two consecutive activities. Long edges = where
#cases are waiting the longest.

print("\n=== STEP 6: Performance analysis (bottlenecks) ===")

perf_dfg, perf_start, perf_end = pm4py.discover_performance_dfg(event_log)

print("Average time between activities (sorted, slowest first):")
for edge, stats in sorted(perf_dfg.items(), key=lambda x: -x[1]["mean"]):
    days = stats["mean"] / 86400
    print(f"  {edge[0]} -> {edge[1]} : avg {days:.2f} days")

pm4py.save_vis_performance_dfg(perf_dfg, perf_start, perf_end, "p2p_performance_dfg.png")


=== STEP 6: Performance analysis (bottlenecks) ===
Average time between activities (sorted, slowest first):
  Make Payment -> Create Purchase Requisition : avg 323.07 days
  Approve Purchase Requisition -> Receive Goods : avg 23.10 days
  Verify Goods Receipt -> Make Payment : avg 17.15 days
  Create Purchase Requisition -> Send Purchase Order to Supplier : avg 14.87 days
  Receive Goods -> Create Invoice : avg 12.38 days
  Verify Goods Receipt -> Approve Invoice : avg 11.47 days
  Create Purchase Requisition -> Create Purchase Order : avg 11.38 days
  Approve Purchase Requisition -> Send Purchase Order to Supplier : avg 11.24 days
  Create Purchase Order -> Receive Goods : avg 10.26 days
  Create Invoice -> Make Payment : avg 9.30 days
  Send Purchase Order to Supplier -> Verify Goods Receipt : avg 7.85 days
  Verify Goods Receipt -> Create Invoice : avg 5.67 days
  Receive Goods -> Verify Goods Receipt : avg 5.11 days
  Approve Purchase Requisition -> Create Purchase Order : avg 4.9

''

In [22]:
#7-CONFORMANCE CHECKING
#Compare the real event log against the discovered model to
# find deviations, cases that skip steps, do extra steps, or do them out of
# order. Two common techniques:
#   (a) Token-based replay, fast, tells you how many tokens were "missing" or
#       "left over" per case (a proxy for how far it deviates from the model)
#   (b) Alignments, slower but precise, tells you exactly WHERE each case
#       deviates from the model

print("\n=== STEP 7: Conformance checking ===")

# --- (a) Token-based replay ---
replay_results = pm4py.conformance_diagnostics_token_based_replay(
    event_log, net, initial_marking, final_marking
)

fit_traces = sum(1 for r in replay_results if r["trace_is_fit"])
print(f"Token-based replay: {fit_traces}/{n_cases} cases perfectly fit the model "
      f"({fit_traces/n_cases*100:.1f}%)")

# Show a couple of non-fitting cases for inspection
non_fitting = [i for i, r in enumerate(replay_results) if not r["trace_is_fit"]]
print(f"Non-fitting cases (first 5 indices): {non_fitting[:5]}")

# --- (b) Alignments (more precise, but slower — sample if log is huge) ---
alignments = pm4py.conformance_diagnostics_alignments(
    event_log, net, initial_marking, final_marking
)
fitness = pm4py.fitness_alignments(event_log, net, initial_marking, final_marking)
print(f"\nAlignment-based fitness (log-level): {fitness}")

# Overall fitness score using pm4py's built-in evaluator
fitness_tbr = pm4py.fitness_token_based_replay(event_log, net, initial_marking, final_marking)
print(f"Token-based replay fitness (log-level): {fitness_tbr}")

print("\nDone. Outputs saved: p2p_dfg.png, p2p_petri_net.png, p2p_performance_dfg.png")


=== STEP 7: Conformance checking ===


replaying log with TBR, completed traces ::   0%|          | 0/34 [00:00<?, ?it/s]

Token-based replay: 934/934 cases perfectly fit the model (100.0%)
Non-fitting cases (first 5 indices): []


aligning log, completed variants ::   0%|          | 0/34 [00:00<?, ?it/s]

aligning log, completed variants ::   0%|          | 0/34 [00:00<?, ?it/s]


Alignment-based fitness (log-level): {'percFitTraces': 100.0, 'averageFitness': 1.0, 'percentage_of_fitting_traces': 100.0, 'average_trace_fitness': 1.0, 'log_fitness': 0.999900002143139}


replaying log with TBR, completed traces ::   0%|          | 0/34 [00:00<?, ?it/s]

Token-based replay fitness (log-level): {'perc_fit_traces': 100.0, 'average_trace_fitness': 1.0, 'log_fitness': 1.0, 'percentage_of_fitting_traces': 100.0}

Done. Outputs saved: p2p_dfg.png, p2p_petri_net.png, p2p_performance_dfg.png


***NOW TO-BE PROCESS***

In [25]:
import pandas as pd
import pm4py

CSV_PATH = "P2P.csv"
df = pd.read_csv(CSV_PATH)
df = df.drop(columns=[c for c in df.columns if "Unnamed" in c], errors="ignore")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values(["Order_ID", "Timestamp"]).reset_index(drop=True)
df = df.rename(columns={
    "Order_ID": "case:concept:name",
    "Activity": "concept:name",
    "Timestamp": "time:timestamp",
})
n_cases = df["case:concept:name"].nunique()

In [26]:
#1-"Approve Invoice" must eventually-precede "Make Payment"
# pm4py.filter_eventually_follows_relation keeps cases where the relation
# A -> ... -> B holds SOMEWHERE in the trace (order preserved, gaps allowed).
# retain=True  -> keep cases that SATISFY the rule (compliant cases)
# retain=False -> keep cases that VIOLATE the rule (the ones you want to flag)

compliant = pm4py.filter_eventually_follows_relation(
    df, [("Approve Invoice", "Make Payment")], retain=True
)
violating = pm4py.filter_eventually_follows_relation(
    df, [("Approve Invoice", "Make Payment")], retain=False
)

n_compliant = compliant["case:concept:name"].nunique()
n_violating = violating["case:concept:name"].nunique()

print("=== RULE: Approve Invoice must precede Make Payment ===")
print(f"Compliant cases: {n_compliant} ({n_compliant/n_cases*100:.1f}%)")
print(f"Violating cases: {n_violating} ({n_violating/n_cases*100:.1f}%)")
print("Sample violating case IDs:", violating["case:concept:name"].unique()[:5].tolist())

#2-Every order must have "Create Purchase Requisition" before
# "Create Purchase Order" (can't buy something with no requisition on file)

violating_2 = pm4py.filter_eventually_follows_relation(
    df, [("Create Purchase Requisition", "Create Purchase Order")], retain=False
)
n_violating_2 = violating_2["case:concept:name"].nunique()

print("\n=== RULE: Requisition must precede Purchase Order ===")
print(f"Violating cases: {n_violating_2} ({n_violating_2/n_cases*100:.1f}%)")

#3-Goods must be received before an invoice is created
# (protects against paying for goods never verified as received)

violating_3 = pm4py.filter_eventually_follows_relation(
    df, [("Receive Goods", "Create Invoice")], retain=False
)
n_violating_3 = violating_3["case:concept:name"].nunique()

print("\n=== RULE: Receive Goods must precede Create Invoice ===")
print(f"Violating cases: {n_violating_3} ({n_violating_3/n_cases*100:.1f}%)")



=== RULE: Approve Invoice must precede Make Payment ===
Compliant cases: 769 (82.3%)
Violating cases: 165 (17.7%)
Sample violating case IDs: ['ORD0004', 'ORD0006', 'ORD0018', 'ORD0021', 'ORD0023']

=== RULE: Requisition must precede Purchase Order ===
Violating cases: 149 (16.0%)

=== RULE: Receive Goods must precede Create Invoice ===
Violating cases: 181 (19.4%)


In [28]:
# Görmek istediğimiz örnek case listesi
ornek_caseler = ['ORD0004', 'ORD0006']

# Veriyi bu iki case için filtreleyip temiz bir şekilde yazdırıyoruz
for case_id in ornek_caseler:
    print(f"\n" + "="*50)
    print(f"ORDER SUMMARY(Case ID): {case_id}")
    print("="*50)

    # İlgili siparişi bul ve zaman sırasına göre getir
    kısıtlı_df = df[df['case:concept:name'] == case_id].sort_values('time:timestamp')

    # Sadece aktivite adını ve gerçekleşme zamanını göster
    for idx, row in kısıtlı_df.iterrows():
        zaman = row['time:timestamp'].strftime('%Y-%m-%d %H:%M')
        print(f"[{zaman}] -> {row['concept:name']}")


ORDER SUMMARY(Case ID): ORD0004
[2024-07-14 19:11] -> Create Purchase Requisition
[2024-07-20 10:43] -> Approve Purchase Requisition
[2024-07-21 09:31] -> Create Purchase Order
[2024-07-27 18:59] -> Send Purchase Order to Supplier
[2024-08-05 16:25] -> Receive Goods
[2024-08-12 17:08] -> Verify Goods Receipt
[2024-08-21 19:23] -> Create Invoice
[2024-08-27 15:42] -> Make Payment

ORDER SUMMARY(Case ID): ORD0006
[2024-02-05 17:28] -> Create Purchase Requisition
[2024-02-09 08:24] -> Approve Purchase Requisition
[2024-02-12 16:43] -> Create Purchase Order
[2024-02-16 15:25] -> Send Purchase Order to Supplier
[2024-02-19 19:36] -> Receive Goods
[2024-02-28 13:36] -> Verify Goods Receipt
[2024-03-08 18:48] -> Create Invoice
[2024-03-18 15:59] -> Make Payment


In [29]:
#SUMMARY: this is the audit-ready output — targeted, explainable violations,
# with none of the false positives a rigid single-sequence model would cause.

print("\n=== SUMMARY ===")
print(f"Total cases: {n_cases}")
print(f"Rule 1 violations (payment without invoice approval): {n_violating} "
      f"({n_violating/n_cases*100:.1f}%)")
print(f"Rule 2 violations (PO without requisition):           {n_violating_2} "
      f"({n_violating_2/n_cases*100:.1f}%)")
print(f"Rule 3 violations (invoice before goods received):    {n_violating_3} "
      f"({n_violating_3/n_cases*100:.1f}%)")


=== SUMMARY ===
Total cases: 934
Rule 1 violations (payment without invoice approval): 165 (17.7%)
Rule 2 violations (PO without requisition):           149 (16.0%)
Rule 3 violations (invoice before goods received):    181 (19.4%)
